#Developing a RAG application using Langchain   

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [45]:
import os
from dotenv import load_dotenv
load_dotenv()

from google import genai
from google.genai import types

from IPython.display import display
from IPython.display import Markdown
import textwrap

from pathlib import Path
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import  GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA

In [23]:
model = "gemini-3.6-flash"
client = genai.Client()

In [26]:
response = client.models.generate_content(
    model=model,
    contents="Explain Generative AI with 3 bullet points")
print(response.text)
Markdown(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Here is Generative AI explained in 3 bullet points:

*   **Creates Brand-New Content:** Unlike traditional AI that only analyzes or categorizes existing data, Generative AI produces entirely new, human-like content—including text, images, music, video, and computer code.
*   **Learns from Massive Patterns:** It works by using advanced deep learning models trained on vast datasets, allowing the AI to understand underlying patterns and predict what should come next when given a prompt.
*   **Powers Modern Everyday Tools:** It is the core technology behind popular applications like ChatGPT (for writing and brainstorming), Midjourney or DALL-E (for art and design), and GitHub Copilot (for programming).


Here is Generative AI explained in 3 bullet points:

*   **Creates Brand-New Content:** Unlike traditional AI that only analyzes or categorizes existing data, Generative AI produces entirely new, human-like content—including text, images, music, video, and computer code.
*   **Learns from Massive Patterns:** It works by using advanced deep learning models trained on vast datasets, allowing the AI to understand underlying patterns and predict what should come next when given a prompt.
*   **Powers Modern Everyday Tools:** It is the core technology behind popular applications like ChatGPT (for writing and brainstorming), Midjourney or DALL-E (for art and design), and GitHub Copilot (for programming).

In [ ]:
hist = model.start_chat()
response = hist.send_message("Hi! Give me a recipe to make a margeritta pizza from scratch.")
Markdown(response.text)

for i in hist.history:
    print(i)
    print('\n\n')
i.parts[0].text 

model.count_tokens("Now please help me find the nearest supermarket from where I can buy the ingredients.")

In [ ]:
def get_response(prompt, generation_config={}):
    response = model.generate_content(contents=prompt, 
    generation_config=generation_config)
    return response

for temp in [0.0, 0.25, 0.5, 0.75, 1.0]:
  config = genai.types.GenerationConfig(temperature=temp)
  result = get_response("Explain the concepts of XGBoost and Random Forest with real-life use cases", generation_config=config)

  print(f"\n\nFor temperature value {temp}, the results are: \n\n")
  display(Markdown(result.text))

In [ ]:
def get_response(prompt, generation_config={}):
    response = model.generate_content(contents=prompt, generation_config=generation_config)
    return response

for m_o_tok in [1000, 2000, 7000]:
    config = genai.types.GenerationConfig(max_output_tokens=m_o_tok)
    
    print(f"\n\n--- Max Output Tokens: {m_o_tok} ---")
    try:
        # Get the result
        result = get_response("Explain the concepts of XGBoost and Random Forest with real-life use cases", generation_config=config)
        
        # Only print text if the model actually returned text parts
        if result.candidates[0].content.parts:
            display(Markdown(result.text))
        # Handle the empty response
        # This is because different models have different MOTok to even start generating the answer
        else: 
            print(f"Result was empty! The model stopped because: {result.candidates[0].finish_reason}")
            print("(Reason 2 means 'MAX_TOKENS' was reached before a complete word could be formed)")
            
    except Exception as e:
        print(f"Skipped due to error: {e}")

In [ ]:
def get_response(prompt, generation_config={}):
    response = model.generate_content(contents=prompt, 
    generation_config=generation_config)
    return response

for k in [1, 4, 16, 32, 40]:
    config = genai.types.GenerationConfig(top_k=k)
    result = get_response("Explain the concepts of XGBoost and Random Forest with real-life use cases", generation_config=config)

    print(f"\n\nFor top k value {k}, the results are: \n\n")
    display(Markdown(result.text))

# Demo for Retrieval Augmented Generation

Step 1: Load a PDF and extract its text

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Initialize the loader with the path to your PDF file
pdfloader = PyPDFLoader("./whatisai.pdf")

# Load the PDF and split it into pages
pdf_pages = pdfloader.load()

print(f"Total pages loaded: {len(pdf_pages)}")

# Declare chunk size and overlap for text splitting
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200


# Splitting text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
context = "\n\n".join(str(p.page_content) for p in pdf_pages)
texts = text_splitter.split_text(context)
print(f"Total chunks created: {len(texts)}")
print(f"First chunk: {texts[0]}")
print(f"Second chunk: {texts[1]}")
print(f"Third chunk: {texts[2]}")

Step 2: Create the Gemini model and generate embeddings

In [ ]:
# Initiate the model and embeddings with the API key from environment variables
API_KEY = os.getenv("GEMINI_API_KEY")

# Create the model and embeddings instances
gemini_model = ChatGoogleGenerativeAI(model=model, google_api_key=API_KEY, temperature=0.8)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", api_key=API_KEY)


Use Chroma for vector Database

In [ ]:
# Create a vector database and a retriever for querying
vector_index = Chroma.from_texts(texts, embeddings)
retriever = vector_index.as_retriever(search_kwargs={"k" : 5})

In [ ]:
# Create a RAG question-answering chain using the model and retriever
qa_chain = RetrievalQA.from_chain_type(gemini_model, retriever=retriever, return_source_documents=True)

In [37]:
# Example usage 
question = "Who can learn AI?"
result = qa_chain.invoke({"query": question})
print("Answer:", result["result"])

Answer: Based on the provided context, there is no information mentioning who can learn AI. The text only specifies what someone should study before or while learning AI.


Use pgVector as vector database

# Use the following scripts to create the table that stores embeddings and create index:

CREATE TABLE documents (
    id SERIAL PRIMARY KEY,
    content TEXT NOT NULL,
    embedding VECTOR(3072),  -- 3072 dimensions for default gemini-embedding-001
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE INDEX ON documents USING ivfflat (embedding vector_cosine_ops)

We use ivfflat index because Postgresql database cannot support more than 2000 dimensions for HNSW index

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("/Users/admin/Documents/rag/RAG_Using_Langchain/.env")

import psycopg
from pgvector.psycopg import register_vector

DATABASE_URL = os.getenv("DATABASE_URL")
COLLECTION_NAME = "documents" # Ensure this matches the table name in your PostgreSQL database

def load_document_to_pgvector(pdf_pages):
    with psycopg.connect(DATABASE_URL) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            # Insert documents and their embeddings into the table
            for doc in pdf_pages:
                embedding_vector = embeddings.embed_query(doc.page_content)
                try:
                    cur.execute(f"""
                        INSERT INTO {COLLECTION_NAME} (content, embedding)
                        VALUES (%s, %s)
                        """, (doc.page_content, embedding_vector))
                except Exception as exc:
                    print(f"Failed to insert document: {exc}")
            conn.commit()


documents


In [51]:
load_document_to_pgvector(pdf_pages)   

In [63]:
def retrieve_relevant_context(user_query, top_k=3):
    SIMILARITY_THRESHOLD = 0.70  # Define a threshold for filtering weak matches
    context_chunks = []
    with psycopg.connect(DATABASE_URL) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            # Embed the user query
            query_embedding = embeddings.embed_query(user_query)
            
            # Retrieve the top_k most relevant documents based on cosine similarity
            cur.execute(f"""
                SELECT content, (1 - (embedding <=> %s::vector)) AS similarity
                FROM documents
                ORDER BY embedding <=> %s::vector ASC
                LIMIT %s;
                """, (query_embedding, query_embedding, top_k))
            
            results = cur.fetchall()

            for idx, (content, similarity) in enumerate(results, 1):
                # Print the score for debugging/monitoring
                print(f"Chunk #{idx} | Similarity Score: {similarity:.4f}")
                print(f"Snippet: {content[:100]}...\n")
                
                # Optional: Filter out weak matches (e.g., lower than 0.60)
                if similarity >= SIMILARITY_THRESHOLD:
                    context_chunks.append(content)
                else:
                    print(f"--> Chunk #{idx} dropped (below similarity threshold)")
    return context_chunks



In [64]:
user_query = "Name few good reading materials for learning AI."

retrieved_contexts = retrieve_relevant_context(user_query, top_k=3)

print("Retrieved Contexts:")
print(retrieved_contexts)
    

Chunk #1 | Similarity Score: 0.7457
Snippet: knowing the languages currently in fashion. In the late 1990 s, these include
C++ and Java.
Q. What ...

Chunk #2 | Similarity Score: 0.6976
Snippet: 9.13
References
[Den98] Daniel Dennett. Brainchildren: Essays on Designing Minds . MIT
Press, 1998.
...

--> Chunk #2 dropped (below similarity threshold)
Chunk #3 | Similarity Score: 0.6853
Snippet: been previous credit card frauds at this establishment).
4 More questions
Q. How is AI research done...

--> Chunk #3 dropped (below similarity threshold)
Retrieved Contexts:
['knowing the languages currently in fashion. In the late 1990 s, these include\nC++ and Java.\nQ. What is a good textbook on AI?\nA. Artiﬁcial Intelligence by Stuart Russell and Peter Norvig, Prentice Hall\nis the most commonly used textbbook in 1997. The general view s expressed\nthere do not exactly correspond to those of this essay. Artiﬁcial Intelligence:\nA New Synthesis by Nils Nilsson, Morgan Kaufman, may be easier to 